In [ ]:
import os
# ============================================================
# CLEAN ORIGINAL-DATA BASELINE COMPARISON
# ============================================================
#
# This notebook:
#   1. Reconstructs Full and Pure pre-final-clipping scores.
#   2. Validates the reconstruction against saved final scores.
#   3. Loads raw anomaly scores for all baseline methods.
#   4. Selects exactly the top 2% from every method.
#   5. Evaluates flood-context overlap.
#   6. Evaluates terrain and hydrological characteristics.
#   7. Evaluates spatial connected components without
#      morphological opening.
#   8. Computes pairwise Jaccard overlap.
#   9. Saves paper-oriented tables to Excel, CSV, and LaTeX.
#
# No model training or anomaly-score recomputation is performed.
# ============================================================


from pathlib import Path
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import label
from openpyxl.styles import Font, Alignment


# ============================================================
# 1. Paths
# ============================================================

ROOT_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

BASELINE_DIR = (
    ROOT_DIR
    / "02_Baseline_Results_Seed_42"
)

PROPOSED_DIR = (
    ROOT_DIR
    / "03_Proposed_Results_Seed_42"
)

OUTPUT_DIR = (
    ROOT_DIR
    / "04_Baseline_Comparison_Results_Seed_42"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Main dataset files
X_PATH = ROOT_DIR / "X.npy"
Y_PATH = ROOT_DIR / "y.npy"

# Use valid_mask_1.npy when available because it was used in
# the previous outlier-analysis notebook.
if (ROOT_DIR / "valid_mask_1.npy").exists():
    VALID_MASK_PATH = (
        ROOT_DIR
        / "valid_mask_1.npy"
    )
else:
    VALID_MASK_PATH = (
        ROOT_DIR
        / "valid_mask.npy"
    )


# Proposed-method parameter files
FULL_SPATIAL_PARAMETER_PATH = (
    PROPOSED_DIR
    / "full_spatial_normalization_params.npz"
)

FULL_PURE_PARAMETER_PATH = (
    PROPOSED_DIR
    / "full_pure_final_normalization_params.npz"
)

FULL_PURE_RELIABILITY_PATH = (
    PROPOSED_DIR
    / "full_pure_reliability_diagnostics.npz"
)

PURE_SPATIAL_SCORE_PATH = (
    PROPOSED_DIR
    / "pure_spatial_score_normalized.npy"
)


# Output files
OUTPUT_EXCEL = (
    OUTPUT_DIR
    / "Clean_Data_Baseline_Comparison_Results.xlsx"
)

OUTPUT_SUMMARY_CSV = (
    OUTPUT_DIR
    / "Overall_Comparison.csv"
)


# ============================================================
# 2. Experimental settings
# ============================================================

ANOMALY_BUDGET = 0.02

METHOD_ORDER = [
    "IF",
    "LOF",
    "MD",
    "PCA",
    "AE",
    "HBOS",
    "Pure",
    "Full",
]


# The first 16 columns are the continuous features in X.npy.
# The remaining seven columns are land-use indicators.
FEATURE_NAMES = [
    "elevation",             # 0
    "slope",                 # 1
    "flow_acc",              # 2
    "profile_curv",          # 3
    "plan_curv",             # 4
    "depression_depth",      # 5
    "local_std",             # 6
    "elev_percentile",       # 7
    "elev_contrast",         # 8
    "slope_variability",     # 9
    "flow_contrast",         # 10
    "road_dist",             # 11
    "river_dist",            # 12
    "ndvi",                  # 13
    "ndwi",                  # 14
    "ndbi",                  # 15
    "landuse_1",             # 16
    "landuse_2",             # 17
    "landuse_3",             # 18
    "landuse_4",             # 19
    "landuse_5",             # 20
    "landuse_6",             # 21
    "landuse_7",             # 22
]

SLOPE_INDEX = FEATURE_NAMES.index(
    "slope"
)

FLOW_ACC_INDEX = FEATURE_NAMES.index(
    "flow_acc"
)


# ============================================================
# 3. Utility functions
# ============================================================

def require_file(path):
    """Stop execution when a required file is unavailable."""

    if not path.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{path}"
        )


def load_vector(
    path,
    expected_length=None,
    name=None,
    mmap_mode="r",
):
    """Load and validate a one-dimensional numeric array."""

    require_file(path)

    array = np.load(
        path,
        mmap_mode=mmap_mode,
    )

    array = np.asarray(
        array
    ).reshape(-1)

    array_name = (
        name
        if name is not None
        else path.name
    )

    if expected_length is not None:
        if array.size != expected_length:
            raise ValueError(
                f"{array_name} contains "
                f"{array.size:,} values; "
                f"{expected_length:,} were expected."
            )

    if not np.issubdtype(
        array.dtype,
        np.number,
    ):
        raise TypeError(
            f"{array_name} is not numeric."
        )

    if not np.isfinite(array).all():
        bad_count = int(
            (~np.isfinite(array)).sum()
        )

        raise ValueError(
            f"{array_name} contains "
            f"{bad_count:,} non-finite values."
        )

    return array


def normalize_with_fixed_bounds(
    values,
    lower_bound,
    upper_bound,
):
    """
    Reproduce the clipping and normalization used in the
    proposed-method pipeline.
    """

    lower_bound = float(
        lower_bound
    )

    upper_bound = float(
        upper_bound
    )

    if upper_bound <= lower_bound:
        raise ValueError(
            "Upper normalization bound must exceed "
            "the lower bound."
        )

    normalized = (
        np.clip(
            values,
            lower_bound,
            upper_bound,
        )
        - lower_bound
    ) / (
        upper_bound
        - lower_bound
    )

    return np.clip(
        normalized,
        0.0,
        1.0,
    ).astype(
        np.float32,
        copy=False,
    )


def exact_top_k_indices(
    scores,
    selected_count,
):
    """
    Select exactly the requested number of highest-scoring
    observations.

    Larger values indicate greater anomalousness.

    Exact ties at the selection boundary are resolved by
    selecting the lower pixel indices first.
    """

    scores = np.asarray(
        scores
    ).reshape(-1)

    n_samples = scores.size

    if not 1 <= selected_count <= n_samples:
        raise ValueError(
            "selected_count must be between 1 and "
            f"{n_samples:,}."
        )

    boundary_position = (
        n_samples
        - selected_count
    )

    boundary_score = np.partition(
        scores,
        boundary_position,
    )[boundary_position]

    above_boundary = np.flatnonzero(
        scores > boundary_score
    )

    boundary_ties = np.flatnonzero(
        scores == boundary_score
    )

    number_required_from_ties = (
        selected_count
        - above_boundary.size
    )

    if number_required_from_ties < 0:
        raise RuntimeError(
            "Unexpected top-k boundary calculation."
        )

    selected_indices = np.concatenate(
        [
            above_boundary,
            boundary_ties[
                :number_required_from_ties
            ],
        ]
    )

    if selected_indices.size != selected_count:
        raise RuntimeError(
            "Exact top-k selection did not return the "
            "required number of observations."
        )

    # Sorted indices are convenient for saving and comparison.
    selected_indices = np.sort(
        selected_indices.astype(
            np.int64,
            copy=False,
        )
    )

    return (
        selected_indices,
        float(boundary_score),
    )


def calculate_jaccard(
    first_mask,
    second_mask,
):
    """Calculate Jaccard overlap between two binary masks."""

    intersection = int(
        np.logical_and(
            first_mask,
            second_mask,
        ).sum()
    )

    union = int(
        np.logical_or(
            first_mask,
            second_mask,
        ).sum()
    )

    if union == 0:
        return np.nan

    return (
        intersection
        / union
    )


def connected_component_statistics(
    one_dimensional_mask,
    valid_mask,
):
    """
    Calculate connected components directly from the original
    selected anomaly mask.

    No binary opening, closing, erosion, or dilation is applied.

    Eight-neighbour connectivity is used.
    """

    one_dimensional_mask = np.asarray(
        one_dimensional_mask,
        dtype=bool,
    ).reshape(-1)

    valid_mask = np.asarray(
        valid_mask,
        dtype=bool,
    )

    expected_valid_pixels = int(
        valid_mask.sum()
    )

    if one_dimensional_mask.size != expected_valid_pixels:
        raise ValueError(
            "The one-dimensional anomaly mask does not match "
            "the number of valid raster pixels."
        )

    anomaly_grid = np.zeros(
        valid_mask.shape,
        dtype=bool,
    )

    anomaly_grid[
        valid_mask
    ] = one_dimensional_mask

    # Eight-neighbour connectivity
    connectivity = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    labelled_grid, number_of_components = label(
        anomaly_grid,
        structure=connectivity,
    )

    component_sizes = np.bincount(
        labelled_grid.ravel()
    )[1:]

    selected_count = int(
        one_dimensional_mask.sum()
    )

    if component_sizes.sum() != selected_count:
        raise RuntimeError(
            "Connected-component sizes do not sum to the "
            "selected anomaly count."
        )

    if number_of_components == 0:
        return {
            "Components": 0,
            "Mean component size": 0.0,
            "Median component size": 0.0,
            "95th-percentile component size": 0.0,
            "Maximum component size": 0,
            "Singleton components": 0,
            "Singleton component ratio (%)": 0.0,
            "Largest component share (%)": 0.0,
            "Components per 1000 anomaly pixels": 0.0,
        }

    singleton_count = int(
        np.sum(
            component_sizes == 1
        )
    )

    return {
        "Components": int(
            number_of_components
        ),
        "Mean component size": float(
            np.mean(component_sizes)
        ),
        "Median component size": float(
            np.median(component_sizes)
        ),
        "95th-percentile component size": float(
            np.percentile(
                component_sizes,
                95,
            )
        ),
        "Maximum component size": int(
            component_sizes.max()
        ),
        "Singleton components": singleton_count,
        "Singleton component ratio (%)": (
            singleton_count
            / number_of_components
            * 100.0
        ),
        "Largest component share (%)": (
            component_sizes.max()
            / selected_count
            * 100.0
        ),
        "Components per 1000 anomaly pixels": (
            number_of_components
            / selected_count
            * 1000.0
        ),
    }


# ============================================================
# 4. Check and load the original dataset
# ============================================================

for required_path in [
    X_PATH,
    Y_PATH,
    VALID_MASK_PATH,
    BASELINE_DIR,
    PROPOSED_DIR,
    FULL_SPATIAL_PARAMETER_PATH,
    FULL_PURE_PARAMETER_PATH,
    FULL_PURE_RELIABILITY_PATH,
    PURE_SPATIAL_SCORE_PATH,
]:
    require_file(
        required_path
    )


X = np.load(
    X_PATH,
    mmap_mode="r",
)

flood_label = load_vector(
    Y_PATH,
    name="Flood/context label",
)

valid_mask = np.load(
    VALID_MASK_PATH
).astype(
    bool,
    copy=False,
)


if X.ndim != 2:
    raise ValueError(
        f"X.npy must be two-dimensional; found {X.shape}."
    )

n_samples, n_features = X.shape

if n_features != len(FEATURE_NAMES):
    raise ValueError(
        f"X.npy contains {n_features} features, whereas "
        f"{len(FEATURE_NAMES)} were expected."
    )

if flood_label.size != n_samples:
    raise ValueError(
        "y.npy does not have the same number of rows as X.npy."
    )

if int(valid_mask.sum()) != n_samples:
    raise ValueError(
        "The number of valid raster pixels does not match "
        "the number of rows in X.npy."
    )


flood_label = (
    flood_label > 0
)

slope_values = X[
    :,
    SLOPE_INDEX,
]

flow_acc_values = X[
    :,
    FLOW_ACC_INDEX,
]


selected_count = int(
    np.ceil(
        ANOMALY_BUDGET
        * n_samples
    )
)


print("Dataset loaded successfully.")
print("Samples:", f"{n_samples:,}")
print("Features:", n_features)
print("Raster shape:", valid_mask.shape)
print("Valid pixels:", f"{int(valid_mask.sum()):,}")
print(
    "Top-2% selected pixels:",
    f"{selected_count:,}",
)
print(
    "Flood-labelled pixels:",
    f"{int(flood_label.sum()):,}",
)




In [ ]:
# ============================================================
# 5. Reconstruct the clean Full pre-final-clipping score
# ============================================================

FULL_COMPONENT_FILES = [
    "ae_score_normalized.npy",
    "if_feature_score_normalized.npy",
    "lof_feature_score_normalized.npy",
    "md_feature_score_normalized.npy",
    "if_embedding_score_normalized.npy",
    "lof_embedding_score_normalized.npy",
    "md_embedding_score_normalized.npy",
]


full_base_score = np.zeros(
    n_samples,
    dtype=np.float32,
)

for component_filename in FULL_COMPONENT_FILES:

    component_path = (
        PROPOSED_DIR
        / component_filename
    )

    component_score = load_vector(
        component_path,
        expected_length=n_samples,
        name=component_filename,
    )

    full_base_score += component_score

    print(
        "Loaded Full component:",
        component_filename,
    )


full_base_score /= float(
    len(FULL_COMPONENT_FILES)
)


spatial_score_normalized = load_vector(
    PROPOSED_DIR
    / "spatial_score_normalized.npy",
    expected_length=n_samples,
    name="Normalized spatial score",
)


with np.load(
    FULL_SPATIAL_PARAMETER_PATH
) as full_spatial_parameters:

    required_full_spatial_keys = [
        "base_weight",
        "spatial_weight",
        "p1",
        "p99",
    ]

    missing_keys = [
        key
        for key in required_full_spatial_keys
        if key not in full_spatial_parameters.files
    ]

    if missing_keys:
        raise KeyError(
            "Missing keys from "
            "full_spatial_normalization_params.npz: "
            + ", ".join(missing_keys)
        )

    full_base_weight = float(
        full_spatial_parameters[
            "base_weight"
        ]
    )

    full_spatial_weight = float(
        full_spatial_parameters[
            "spatial_weight"
        ]
    )

    full_spatial_p1 = float(
        full_spatial_parameters[
            "p1"
        ]
    )

    full_spatial_p99 = float(
        full_spatial_parameters[
            "p99"
        ]
    )


full_spatial_before_normalization = (
    full_base_weight
    * full_base_score
    +
    full_spatial_weight
    * spatial_score_normalized
).astype(
    np.float32,
    copy=False,
)


full_spatial_normalized = (
    normalize_with_fixed_bounds(
        full_spatial_before_normalization,
        full_spatial_p1,
        full_spatial_p99,
    )
)


with np.load(
    FULL_PURE_RELIABILITY_PATH
) as reliability_file:

    if "full_reliability" not in reliability_file.files:
        raise KeyError(
            "'full_reliability' was not found in "
            "full_pure_reliability_diagnostics.npz.\n"
            f"Available keys: {reliability_file.files}"
        )

    full_reliability = np.asarray(
        reliability_file[
            "full_reliability"
        ]
    ).reshape(-1)


if full_reliability.size != n_samples:
    raise ValueError(
        "Full reliability has an incorrect length."
    )

if not np.isfinite(
    full_reliability
).all():
    raise ValueError(
        "Full reliability contains non-finite values."
    )


full_final_preclip = (
    full_spatial_normalized
    * full_reliability
).astype(
    np.float32,
    copy=False,
)


FULL_PRECLIP_PATH = (
    OUTPUT_DIR
    / "Full_final_fused_score_preclip.npy"
)


# ------------------------------------------------------------
# Reuse the existing Full preclip file when it is identical
# to the score reconstructed in the current run.
# ------------------------------------------------------------

if FULL_PRECLIP_PATH.exists():

    try:
        existing_full_preclip = np.load(
            FULL_PRECLIP_PATH,
            mmap_mode=None,
        )

    except OSError as exc:
        raise OSError(
            "The existing Full preclip file cannot even be "
            "opened for reading because another process is "
            "holding an exclusive lock.\n"
            f"File: {FULL_PRECLIP_PATH}"
        ) from exc

    if (
        existing_full_preclip.shape
        != full_final_preclip.shape
    ):
        raise ValueError(
            "The existing Full preclip file has a different "
            "shape from the reconstructed Full score."
        )

    full_existing_max_difference = float(
        np.max(
            np.abs(
                existing_full_preclip.astype(
                    np.float64,
                    copy=False,
                )
                -
                full_final_preclip.astype(
                    np.float64,
                    copy=False,
                )
            )
        )
    )

    full_existing_matches = bool(
        np.allclose(
            existing_full_preclip,
            full_final_preclip,
            rtol=1e-7,
            atol=1e-7,
        )
    )

    if not full_existing_matches:
        raise ValueError(
            "The existing Full preclip file does not match "
            "the Full score reconstructed in this run.\n"
            f"Maximum difference: "
            f"{full_existing_max_difference:.10f}\n"
            "Do not reuse or overwrite it until the file lock "
            "has been released."
        )

    print(
        "\nExisting Full pre-final-clipping score is valid "
        "and has been reused."
    )
    print("File:", FULL_PRECLIP_PATH)
    print(
        "Maximum difference:",
        full_existing_max_difference,
    )

    del existing_full_preclip

else:

    np.save(
        FULL_PRECLIP_PATH,
        full_final_preclip,
    )

    print(
        "\nFull pre-final-clipping score reconstructed "
        "and saved."
    )
    print("Saved:", FULL_PRECLIP_PATH)


print("\nFull pre-final-clipping score reconstructed.")
print("Saved:", FULL_PRECLIP_PATH)


# Release arrays that are no longer needed.
del full_base_score
del full_spatial_before_normalization
del full_spatial_normalized
del full_reliability

gc.collect()




In [ ]:
# ============================================================
# 6. Reconstruct the clean Pure pre-final-clipping score
# ============================================================

with np.load(
    FULL_PURE_PARAMETER_PATH
) as full_pure_parameters:

    required_full_pure_keys = [
        "pure_base_weight",
        "pure_spatial_weight",
        "pure_spatial_p1",
        "pure_spatial_p99",
        "full_final_p1",
        "full_final_p99",
        "pure_final_p1",
        "pure_final_p99",
    ]

    missing_keys = [
        key
        for key in required_full_pure_keys
        if key not in full_pure_parameters.files
    ]

    if missing_keys:
        raise KeyError(
            "Missing keys from "
            "full_pure_final_normalization_params.npz: "
            + ", ".join(missing_keys)
        )

    pure_base_weight = float(
        full_pure_parameters[
            "pure_base_weight"
        ]
    )

    pure_spatial_weight = float(
        full_pure_parameters[
            "pure_spatial_weight"
        ]
    )

    pure_spatial_p1 = float(
        full_pure_parameters[
            "pure_spatial_p1"
        ]
    )

    pure_spatial_p99 = float(
        full_pure_parameters[
            "pure_spatial_p99"
        ]
    )

    full_final_p1 = float(
        full_pure_parameters[
            "full_final_p1"
        ]
    )

    full_final_p99 = float(
        full_pure_parameters[
            "full_final_p99"
        ]
    )

    pure_final_p1 = float(
        full_pure_parameters[
            "pure_final_p1"
        ]
    )

    pure_final_p99 = float(
        full_pure_parameters[
            "pure_final_p99"
        ]
    )


ae_score_normalized = load_vector(
    PROPOSED_DIR
    / "ae_score_normalized.npy",
    expected_length=n_samples,
    name="Normalized AE score",
)


# ============================================================
# NEW: Load Pure-specific spatial score.
# This replaces the Full spatial_score_normalized signal.
# ============================================================

pure_spatial_score_normalized = load_vector(
    PURE_SPATIAL_SCORE_PATH,
    expected_length=n_samples,
    name="Normalized Pure-specific spatial score",
)


# ============================================================
# Reconstruct Pure spatially refined score.
#
# Corrected Pure:
#   0.75 * AE
#   +
#   0.25 * Pure-specific spatial signal
#
# The actual values are read from the saved parameter file.
# ============================================================

pure_spatial_before_normalization = (
    pure_base_weight
    * ae_score_normalized
    +
    pure_spatial_weight
    * pure_spatial_score_normalized    # NEW
).astype(
    np.float32,
    copy=False,
)


pure_spatial_normalized = (
    normalize_with_fixed_bounds(
        pure_spatial_before_normalization,
        pure_spatial_p1,
        pure_spatial_p99,
    )
)


with np.load(
    FULL_PURE_RELIABILITY_PATH
) as reliability_file:

    if "pure_reliability" not in reliability_file.files:
        raise KeyError(
            "'pure_reliability' was not found in "
            "full_pure_reliability_diagnostics.npz.\n"
            f"Available keys: {reliability_file.files}"
        )

    pure_reliability = np.asarray(
        reliability_file[
            "pure_reliability"
        ]
    ).reshape(-1)


if pure_reliability.size != n_samples:
    raise ValueError(
        "Pure reliability has an incorrect length."
    )

if not np.isfinite(
    pure_reliability
).all():
    raise ValueError(
        "Pure reliability contains non-finite values."
    )


pure_final_preclip = (
    pure_spatial_normalized
    * pure_reliability
).astype(
    np.float32,
    copy=False,
)


PURE_PRECLIP_PATH = (
    OUTPUT_DIR
    / "Pure_final_fused_score_preclip.npy"
)

np.save(
    PURE_PRECLIP_PATH,
    pure_final_preclip,
)


print("\nPure pre-final-clipping score reconstructed.")
print("Saved:", PURE_PRECLIP_PATH)

# Useful verification only; does not change results.
print(
    "Pure fusion weights:",
    pure_base_weight,
    pure_spatial_weight,
)


del pure_spatial_before_normalization
del pure_spatial_normalized
del pure_spatial_score_normalized    # NEW
del pure_reliability
del ae_score_normalized
del spatial_score_normalized

gc.collect()


# ============================================================
# 7. Validate Full and Pure reconstruction
# ============================================================

saved_full_final_normalized = load_vector(
    PROPOSED_DIR
    / "full_final_anomaly_score.npy",
    expected_length=n_samples,
    name="Saved normalized Full final score",
)

saved_pure_final_normalized = load_vector(
    PROPOSED_DIR
    / "pure_final_anomaly_score.npy",
    expected_length=n_samples,
    name="Saved normalized Pure final score",
)


reconstructed_full_normalized = (
    normalize_with_fixed_bounds(
        full_final_preclip,
        full_final_p1,
        full_final_p99,
    )
)

reconstructed_pure_normalized = (
    normalize_with_fixed_bounds(
        pure_final_preclip,
        pure_final_p1,
        pure_final_p99,
    )
)


full_maximum_difference = float(
    np.max(
        np.abs(
            reconstructed_full_normalized
            - saved_full_final_normalized
        )
    )
)

pure_maximum_difference = float(
    np.max(
        np.abs(
            reconstructed_pure_normalized
            - saved_pure_final_normalized
        )
    )
)


full_validation_passed = bool(
    np.allclose(
        reconstructed_full_normalized,
        saved_full_final_normalized,
        rtol=1e-6,
        atol=1e-6,
    )
)

pure_validation_passed = bool(
    np.allclose(
        reconstructed_pure_normalized,
        saved_pure_final_normalized,
        rtol=1e-6,
        atol=1e-6,
    )
)


reconstruction_validation_df = pd.DataFrame(
    [
        {
            "Method": "Full",
            "Validation passed": full_validation_passed,
            "Maximum absolute difference": (
                full_maximum_difference
            ),
            "Reconstructed score file": str(
                FULL_PRECLIP_PATH
            ),
        },
        {
            "Method": "Pure",
            "Validation passed": pure_validation_passed,
            "Maximum absolute difference": (
                pure_maximum_difference
            ),
            "Reconstructed score file": str(
                PURE_PRECLIP_PATH
            ),
        },
    ]
)


print("\nReconstruction validation:")
print(reconstruction_validation_df)


if not full_validation_passed:
    raise ValueError(
        "Full reconstruction failed validation. "
        "Do not continue with the analysis."
    )

if not pure_validation_passed:
    raise ValueError(
        "Pure reconstruction failed validation. "
        "Do not continue with the analysis."
    )


del reconstructed_full_normalized
del reconstructed_pure_normalized
del saved_full_final_normalized
del saved_pure_final_normalized

gc.collect()

In [ ]:
# ============================================================
# 8. Define the native anomaly-score files
# ============================================================

SCORE_PATHS = {
    "IF": (
        BASELINE_DIR
        / "Baseline_IF_scores_raw.npy"
    ),
    "LOF": (
        BASELINE_DIR
        / "Baseline_LOF_scores_raw.npy"
    ),
    "MD": (
        BASELINE_DIR
        / "Baseline_MD_scores_raw.npy"
    ),
    "PCA": (
        BASELINE_DIR
        / "Baseline_PCA_scores_raw.npy"
    ),
    "AE": (
        BASELINE_DIR
        / "Baseline_AE_scores_raw.npy"
    ),
    "HBOS": (
        BASELINE_DIR
        / "Baseline_HBOS_scores_raw.npy"
    ),
    "Pure": PURE_PRECLIP_PATH,
    "Full": FULL_PRECLIP_PATH,
}


for method_name, score_path in SCORE_PATHS.items():
    require_file(
        score_path
    )

    print(
        f"{method_name:5s}: {score_path.name}"
    )


# ============================================================
# 9. Generate exact top-2% anomaly masks
# ============================================================

method_masks = {}
selection_rows = []


for method_name in METHOD_ORDER:

    print(
        "\nSelecting exact top 2% for:",
        method_name,
    )

    method_score = load_vector(
        SCORE_PATHS[
            method_name
        ],
        expected_length=n_samples,
        name=f"{method_name} anomaly score",
        mmap_mode=(
            None
            if method_name in ["Full", "Pure"]
            else "r"
        ),
    )

    selected_indices, score_threshold = (
        exact_top_k_indices(
            scores=method_score,
            selected_count=selected_count,
        )
    )

    anomaly_mask = np.zeros(
        n_samples,
        dtype=bool,
    )

    anomaly_mask[
        selected_indices
    ] = True

    if int(anomaly_mask.sum()) != selected_count:
        raise RuntimeError(
            f"{method_name} did not select exactly "
            f"{selected_count:,} samples."
        )

    method_masks[
        method_name
    ] = anomaly_mask

    mask_output_path = (
        OUTPUT_DIR
        / f"{method_name}_top2pct_anomaly_mask.npy"
    )

    index_output_path = (
        OUTPUT_DIR
        / f"{method_name}_top2pct_selected_indices.npy"
    )

    np.save(
        mask_output_path,
        anomaly_mask,
    )

    np.save(
        index_output_path,
        selected_indices,
    )

    selection_rows.append(
        {
            "Method": method_name,
            "Selected pixels": selected_count,
            "Anomaly ratio (%)": (
                selected_count
                / n_samples
                * 100.0
            ),
            "Selection threshold": score_threshold,
            "Score source": str(
                SCORE_PATHS[
                    method_name
                ]
            ),
            "Mask file": str(
                mask_output_path
            ),
            "Selected-index file": str(
                index_output_path
            ),
        }
    )


selection_df = pd.DataFrame(
    selection_rows
)


# ============================================================
# 10. Compare new masks with the previously saved masks
#     This sheet is for auditing, not necessarily for the paper.
# ============================================================

OLD_MASK_PATHS = {
    "IF": (
        BASELINE_DIR
        / "Baseline_IF_anomaly_labels.npy"
    ),
    "LOF": (
        BASELINE_DIR
        / "Baseline_LOF_anomaly_labels.npy"
    ),
    "MD": (
        BASELINE_DIR
        / "Baseline_MD_anomaly_labels.npy"
    ),
    "PCA": (
        BASELINE_DIR
        / "Baseline_PCA_anomaly_labels.npy"
    ),
    "AE": (
        BASELINE_DIR
        / "Baseline_AE_anomaly_labels.npy"
    ),
    "HBOS": (
        BASELINE_DIR
        / "Baseline_HBOS_anomaly_labels.npy"
    ),
    "Pure": (
        PROPOSED_DIR
        / "pure_final_anomaly_labels.npy"
    ),
    "Full": (
        PROPOSED_DIR
        / "full_final_anomaly_labels.npy"
    ),
}


selection_audit_rows = []


for method_name in METHOD_ORDER:

    old_mask_path = OLD_MASK_PATHS[
        method_name
    ]

    if not old_mask_path.exists():
        selection_audit_rows.append(
            {
                "Method": method_name,
                "Old mask available": False,
                "Old selected pixels": np.nan,
                "New selected pixels": selected_count,
                "Intersection": np.nan,
                "Union": np.nan,
                "Jaccard": np.nan,
            }
        )

        continue

    old_mask = load_vector(
        old_mask_path,
        expected_length=n_samples,
        name=f"Old {method_name} mask",
    ).astype(
        bool,
        copy=False,
    )

    new_mask = method_masks[
        method_name
    ]

    intersection = int(
        np.logical_and(
            old_mask,
            new_mask,
        ).sum()
    )

    union = int(
        np.logical_or(
            old_mask,
            new_mask,
        ).sum()
    )

    selection_audit_rows.append(
        {
            "Method": method_name,
            "Old mask available": True,
            "Old selected pixels": int(
                old_mask.sum()
            ),
            "New selected pixels": int(
                new_mask.sum()
            ),
            "Intersection": intersection,
            "Union": union,
            "Jaccard": (
                intersection
                / union
                if union > 0
                else np.nan
            ),
        }
    )


selection_audit_df = pd.DataFrame(
    selection_audit_rows
)




In [ ]:
# ============================================================
# 11. Overall comparison and flood-context overlap
# ============================================================

total_flood_pixels = int(
    flood_label.sum()
)

background_flood_ratio = (
    total_flood_pixels
    / n_samples
    * 100.0
)


overall_rows = []


for method_name in METHOD_ORDER:

    anomaly_mask = method_masks[
        method_name
    ]

    anomaly_count = int(
        anomaly_mask.sum()
    )

    flood_overlap_count = int(
        np.logical_and(
            anomaly_mask,
            flood_label,
        ).sum()
    )

    flood_overlap_ratio = (
        flood_overlap_count
        / anomaly_count
        * 100.0
    )

    flood_coverage_ratio = (
        flood_overlap_count
        / total_flood_pixels
        * 100.0
        if total_flood_pixels > 0
        else np.nan
    )

    flood_enrichment = (
        flood_overlap_ratio
        / background_flood_ratio
        if background_flood_ratio > 0
        else np.nan
    )

    overall_rows.append(
        {
            "Method": method_name,
            "Selected anomaly pixels": anomaly_count,
            "Anomaly ratio (%)": (
                anomaly_count
                / n_samples
                * 100.0
            ),
            "Flood-overlap pixels": (
                flood_overlap_count
            ),
            "Flood overlap within anomalies (%)": (
                flood_overlap_ratio
            ),
            "Flood-labelled area recovered (%)": (
                flood_coverage_ratio
            ),
            "Flood-overlap enrichment (x)": (
                flood_enrichment
            ),
        }
    )


overall_df = pd.DataFrame(
    overall_rows
)


print("\nOverall comparison:")
print(
    overall_df.round(3)
)




In [ ]:
# ============================================================
# 12. Terrain and hydrological characteristics
# ============================================================

total_slope_sum = float(
    np.sum(
        slope_values,
        dtype=np.float64,
    )
)

total_flow_sum = float(
    np.sum(
        flow_acc_values,
        dtype=np.float64,
    )
)


terrain_hydrology_rows = []


for method_name in METHOD_ORDER:

    anomaly_mask = method_masks[
        method_name
    ]

    anomaly_count = int(
        anomaly_mask.sum()
    )

    normal_count = (
        n_samples
        - anomaly_count
    )

    selected_slope_values = np.asarray(
        slope_values[
            anomaly_mask
        ]
    )

    selected_flow_values = np.asarray(
        flow_acc_values[
            anomaly_mask
        ]
    )

    selected_slope_sum = float(
        np.sum(
            selected_slope_values,
            dtype=np.float64,
        )
    )

    selected_flow_sum = float(
        np.sum(
            selected_flow_values,
            dtype=np.float64,
        )
    )

    mean_slope_anomalies = (
        selected_slope_sum
        / anomaly_count
    )

    mean_slope_normal = (
        total_slope_sum
        - selected_slope_sum
    ) / normal_count

    mean_flow_anomalies = (
        selected_flow_sum
        / anomaly_count
    )

    mean_flow_normal = (
        total_flow_sum
        - selected_flow_sum
    ) / normal_count

    terrain_hydrology_rows.append(
        {
            "Method": method_name,
            "Mean slope in anomalies": (
                mean_slope_anomalies
            ),
            "Median slope in anomalies": float(
                np.median(
                    selected_slope_values
                )
            ),
            "Mean slope in non-selected areas": (
                mean_slope_normal
            ),
            "Slope mean difference": (
                mean_slope_anomalies
                - mean_slope_normal
            ),
            "Mean flow accumulation in anomalies": (
                mean_flow_anomalies
            ),
            "Median flow accumulation in anomalies": float(
                np.median(
                    selected_flow_values
                )
            ),
            "Mean flow accumulation in non-selected areas": (
                mean_flow_normal
            ),
            "Flow-accumulation mean difference": (
                mean_flow_anomalies
                - mean_flow_normal
            ),
        }
    )

    del selected_slope_values
    del selected_flow_values


terrain_hydrology_df = pd.DataFrame(
    terrain_hydrology_rows
)


print("\nTerrain and hydrological characteristics:")
print(
    terrain_hydrology_df.round(3)
)




In [ ]:
# ============================================================
# 13. Spatial connected-component analysis
# ============================================================

connected_component_rows = []


for method_name in METHOD_ORDER:

    print(
        "Connected-component analysis:",
        method_name,
    )

    component_result = (
        connected_component_statistics(
            one_dimensional_mask=method_masks[
                method_name
            ],
            valid_mask=valid_mask,
        )
    )

    component_result[
        "Method"
    ] = method_name

    connected_component_rows.append(
        component_result
    )

    gc.collect()


connected_component_df = pd.DataFrame(
    connected_component_rows
)


connected_component_df = connected_component_df[
    [
        "Method",
        "Components",
        "Mean component size",
        "Median component size",
        "95th-percentile component size",
        "Maximum component size",
        "Singleton components",
        "Singleton component ratio (%)",
        "Largest component share (%)",
        "Components per 1000 anomaly pixels",
    ]
]


print("\nConnected-component analysis:")
print(
    connected_component_df.round(3)
)




In [ ]:
# ============================================================
# 14. Pairwise Jaccard overlap
# ============================================================

jaccard_matrix = np.zeros(
    (
        len(METHOD_ORDER),
        len(METHOD_ORDER),
    ),
    dtype=np.float64,
)


for row_index, first_method in enumerate(
    METHOD_ORDER
):

    for column_index, second_method in enumerate(
        METHOD_ORDER
    ):

        jaccard_matrix[
            row_index,
            column_index,
        ] = calculate_jaccard(
            method_masks[
                first_method
            ],
            method_masks[
                second_method
            ],
        )


jaccard_df = pd.DataFrame(
    jaccard_matrix,
    index=METHOD_ORDER,
    columns=METHOD_ORDER,
)


print("\nPairwise Jaccard overlap:")
print(
    jaccard_df.round(3)
)




In [ ]:
# ============================================================
# 15. Create paper-oriented tables
# ============================================================

overall_paper_df = overall_df[
    [
        "Method",
        "Flood-overlap pixels",
        "Flood overlap within anomalies (%)",
        "Flood-labelled area recovered (%)",
        "Flood-overlap enrichment (x)",
    ]
].copy()


terrain_paper_df = terrain_hydrology_df[
    [
        "Method",
        "Mean slope in anomalies",
        "Mean slope in non-selected areas",
        "Mean flow accumulation in anomalies",
        "Mean flow accumulation in non-selected areas",
    ]
].copy()


components_paper_df = connected_component_df[
    [
        "Method",
        "Components",
        "Mean component size",
        "Median component size",
        "Maximum component size",
        "Singleton component ratio (%)",
    ]
].copy()


# ============================================================
# 16. Save CSV files
# ============================================================

overall_df.to_csv(
    OUTPUT_SUMMARY_CSV,
    index=False,
)

terrain_hydrology_df.to_csv(
    OUTPUT_DIR
    / "Terrain_Hydrological_Characteristics.csv",
    index=False,
)

connected_component_df.to_csv(
    OUTPUT_DIR
    / "Connected_Component_Analysis.csv",
    index=False,
)

jaccard_df.to_csv(
    OUTPUT_DIR
    / "Pairwise_Jaccard_Overlap.csv",
)


# ============================================================
# 17. Save LaTeX table files
# ============================================================

with open(
    OUTPUT_DIR
    / "table_overall_comparison.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        overall_paper_df.to_latex(
            index=False,
            float_format="%.2f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_terrain_hydrology.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        terrain_paper_df.to_latex(
            index=False,
            float_format="%.3f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_connected_components.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        components_paper_df.to_latex(
            index=False,
            float_format="%.2f",
            escape=True,
        )
    )




In [ ]:
# ============================================================
# 18. Generate figures
# ============================================================

# ------------------------------------------------------------
# Figure 1: Flood overlap
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.bar(
    overall_df["Method"],
    overall_df[
        "Flood overlap within anomalies (%)"
    ],
)

plt.xlabel(
    "Method"
)

plt.ylabel(
    "Flood overlap within selected anomalies (%)"
)

plt.title(
    "Flood-context overlap at the fixed top-2% anomaly budget"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Flood_Overlap_Comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Figure 2: Mean slope
# ------------------------------------------------------------

x_positions = np.arange(
    len(METHOD_ORDER)
)

bar_width = 0.38

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    x_positions - bar_width / 2,
    terrain_hydrology_df[
        "Mean slope in anomalies"
    ],
    width=bar_width,
    label="Selected anomalies",
)

plt.bar(
    x_positions + bar_width / 2,
    terrain_hydrology_df[
        "Mean slope in non-selected areas"
    ],
    width=bar_width,
    label="Non-selected areas",
)

plt.xticks(
    x_positions,
    METHOD_ORDER,
)

plt.xlabel(
    "Method"
)

plt.ylabel(
    "Mean slope"
)

plt.title(
    "Slope characteristics of selected and non-selected pixels"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Slope_Characteristics.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Figure 3: Mean flow accumulation
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    x_positions - bar_width / 2,
    terrain_hydrology_df[
        "Mean flow accumulation in anomalies"
    ],
    width=bar_width,
    label="Selected anomalies",
)

plt.bar(
    x_positions + bar_width / 2,
    terrain_hydrology_df[
        "Mean flow accumulation in non-selected areas"
    ],
    width=bar_width,
    label="Non-selected areas",
)

plt.xticks(
    x_positions,
    METHOD_ORDER,
)

plt.xlabel(
    "Method"
)

plt.ylabel(
    "Mean flow-accumulation feature value"
)

plt.title(
    "Hydrological characteristics of selected and "
    "non-selected pixels"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Flow_Accumulation_Characteristics.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Figure 4: Connected components
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)

plt.bar(
    connected_component_df[
        "Method"
    ],
    connected_component_df[
        "Components"
    ],
)

plt.xlabel(
    "Method"
)

plt.ylabel(
    "Number of connected components"
)

plt.title(
    "Spatial fragmentation of top-2% anomaly masks"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Connected_Component_Counts.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Figure 5: Pairwise Jaccard matrix
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 7)
)

image = plt.imshow(
    jaccard_matrix,
    vmin=0.0,
    vmax=1.0,
)

plt.colorbar(
    image,
    label="Jaccard similarity",
)

plt.xticks(
    np.arange(
        len(METHOD_ORDER)
    ),
    METHOD_ORDER,
    rotation=45,
    ha="right",
)

plt.yticks(
    np.arange(
        len(METHOD_ORDER)
    ),
    METHOD_ORDER,
)

plt.title(
    "Pairwise overlap of top-2% anomaly sets"
)

for row_index in range(
    len(METHOD_ORDER)
):

    for column_index in range(
        len(METHOD_ORDER)
    ):

        plt.text(
            column_index,
            row_index,
            f"{jaccard_matrix[row_index, column_index]:.2f}",
            ha="center",
            va="center",
        )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Pairwise_Jaccard_Matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()




In [ ]:
# ============================================================
# 19. Experimental settings and interpretation notes
# ============================================================

settings_df = pd.DataFrame(
    {
        "Setting": [
            "Dataset",
            "Number of valid pixels",
            "Number of features",
            "Selection budget",
            "Selected pixels per method",
            "Baseline ranking scores",
            "Full ranking score",
            "Pure ranking score",
            "Selection procedure",
            "Connected-component connectivity",
            "Morphological preprocessing",
            "Flood label usage",
        ],
        "Value": [
            "Original uninjected geospatial dataset",
            n_samples,
            n_features,
            "Top 2%",
            selected_count,
            "Raw anomaly scores",
            (
                "Final fused anomaly score before final "
                "clipping and output normalization"
            ),
            (
                "Final fused anomaly score before final "
                "clipping and output normalization"
            ),
            (
                "Deterministic exact top-k; lower pixel "
                "index resolves exact boundary ties"
            ),
            "Eight-neighbour",
            "None",
            (
                "Contextual reference only; not complete "
                "anomaly ground truth"
            ),
        ],
    }
)


paper_use_notes_df = pd.DataFrame(
    {
        "Result": [
            "Overall comparison",
            "Flood overlap",
            "Terrain/hydrology",
            "Connected components",
            "Pairwise Jaccard",
            "Score magnitudes",
            "Score separation",
        ],
        "Interpretation": [
            (
                "Every method selects exactly the same number "
                "of pixels; anomaly count is controlled and "
                "must not be presented as an achievement."
            ),
            (
                "Flood overlap indicates contextual alignment, "
                "not complete detection accuracy."
            ),
            (
                "The values describe the physical conditions "
                "prioritized by each method; larger values are "
                "not universally better."
            ),
            (
                "Calculated directly from the original top-2% "
                "masks without binary opening."
            ),
            (
                "Shows similarity or complementarity between "
                "the regions selected by different methods."
            ),
            (
                "Raw numerical anomaly-score values must not "
                "be directly compared across methods."
            ),
            (
                "Not used as a standalone validation result "
                "because selection is performed using the same "
                "score and score scales differ across methods."
            ),
        ],
    }
)


# ============================================================
# 20. Save final Excel workbook
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

    reconstruction_validation_df.to_excel(
        writer,
        sheet_name="Reconstruction Validation",
        index=False,
    )

    selection_df.to_excel(
        writer,
        sheet_name="Selection Details",
        index=False,
    )

    selection_audit_df.to_excel(
        writer,
        sheet_name="Selection Audit",
        index=False,
    )

    overall_df.to_excel(
        writer,
        sheet_name="Overall Comparison",
        index=False,
    )

    terrain_hydrology_df.to_excel(
        writer,
        sheet_name="Terrain Hydrology",
        index=False,
    )

    connected_component_df.to_excel(
        writer,
        sheet_name="Connected Components",
        index=False,
    )

    jaccard_df.to_excel(
        writer,
        sheet_name="Pairwise Jaccard",
        index=True,
    )

    settings_df.to_excel(
        writer,
        sheet_name="Experimental Settings",
        index=False,
    )

    paper_use_notes_df.to_excel(
        writer,
        sheet_name="Paper Use Guide",
        index=False,
    )

    workbook = writer.book

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for header_cell in worksheet[1]:
            header_cell.font = Font(
                bold=True
            )

            header_cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0].column_letter
            )

            maximum_length = max(
                len(
                    str(cell.value)
                )
                if cell.value is not None
                else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                maximum_length + 2,
                50,
            )


print("\n" + "=" * 72)
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 72)

print("\nExcel workbook:")
print(OUTPUT_EXCEL)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nGenerated result groups:")
print("1. Reconstruction validation")
print("2. Exact top-2% selection")
print("3. Overall flood-context comparison")
print("4. Terrain and hydrological characteristics")
print("5. Connected-component analysis")
print("6. Pairwise Jaccard overlap")
print("7. Paper-oriented LaTeX tables")
print("8. Publication-resolution figures")